# Algoritmos de optimización - Seminario<br>
Nombre y Apellidos: **Carlos Jurado Zalaya, Irune Urrutia Andres**   <br>
Url: https://github.com/jzalaya/grupo-algoritmos<br>
Problema:
> 1. ~~Sesiones de doblaje~~ <br>
>2. ~~Organizar los horarios de partidos de La Liga~~<br>
>3. Combinar cifras y operaciones

Descripción del problema:
* Disponemos de las 9 cifras del 1 al 9 (excluimos el cero) y de los 4 signos básicos de las operaciones fundamentales: suma(+), resta(-), multiplicación(*) y división(/)

* Debemos combinarlos alternativamente sin repetir ninguno de ellos para obtener una cantidad dada. Un ejemplo sería para obtener el 4:
4+2-6/3*1 = 4

* Debe analizarse el problema para encontrar todos los valores enteros posibles planteando las siguientes cuestiones:
  -¿Qué valor máximo y mínimo se pueden obtener según las condiciones del problema?

  -¿Es posible encontrar todos los valores enteros posibles entre dicho mínimo y máximo ?

Nota: Es posible usar la función de python “eval” para evaluar una expresión.

(*) La respuesta es obligatoria





                                        

In [1]:
import itertools

(*)¿Cuantas posibilidades hay sin tener en cuenta las restricciones?<br>



¿Cuantas posibilidades hay teniendo en cuenta todas las restricciones.




Respuesta

Antes de aplicar la restricción de que el resultado de la expresión sea un número entero, pero respetando las restricciones estructurales del problema (utilizar 5 cifras distintas de entre las 9 disponibles y emplear exactamente una vez cada uno de los 4 operadores), el número de expresiones candidatas viene dado por el producto de:

* Las variaciones sin repetición de 9 cifras tomadas de 5 en 5:

$$
V(9,5)=\frac{9!}{(9-5)!}=9\cdot8\cdot7\cdot6\cdot5=15.120
$$

* Las permutaciones sin repetición de los 4 operadores:

$$
P(4)=4!=24
$$

Por tanto, el número total de expresiones posibles es:

$$
V(9,5)\cdot4!=15,120\cdot24=362.880
$$

Se demuestra a continuación:


In [2]:
def generar_expresiones_candidatas():

    cifras = ('1','2','3','4','5','6','7','8','9')                        # Definimos las cifras y operadores posibles
    operaciones = ('+','-','*','/')

    expresiones = []

    for secuencia_cifras in itertools.permutations(cifras, 5):            # Genera todas las variaciones ordenadas de 5 cifras distintas elegidas entre las 9 (sin repetir ninguna)
        for secuencia_operaciones in itertools.permutations(operaciones): # Genera las 4! = 24 formas de ordenar los 4 operadores (aquí se usan siempre los 4, sin repetir)

            expresion = ""                                                # Empezamos la expresión en blanco

            for cifra, operador in zip(secuencia_cifras[:-1],
                                        secuencia_operaciones):           # Son las 4 primeras cifras (todas menos la última) zip() las empareja una a una con cada operador
                expresion += cifra + operador                             # Vamos concatenando "cifra" seguida de "operador"

            expresion += secuencia_cifras[-1]                             # Añadimos la última cifra, que cierra la expresión (no lleva operador detrás)

            expresiones.append(expresion)                                 # Guardamos la expresión completa

    return expresiones                                                    # Devolvemos la lista con las expresiones generadas

In [3]:
expresiones = generar_expresiones_candidatas()                            # Llamamos a la función y guardamos la lista completa de expresiones generadas

print(f"Sin tener en cuenta las restricciones hay {len(expresiones)} combinaciones posibles.") # len() cuenta cuántos elementos tiene la lista (cuántas expresiones se generaron)

Sin tener en cuenta las restricciones hay 362880 combinaciones posibles.


Sobre las 362.880 expresiones
posibles (5 cifras distintas + 4 operadores sin repetir), filtramos las que producen un resultado entero. Como hay cuatro operadores y se usan siempre los cuatro y sin repetición, todas las expresiones contienen exactamente una división, por lo que el resultado siempre es de tipo float y podemos usar is_integer() de forma segura. De las 362.880 expresiones posibles, 90.000  dan como resultado un número entero. Se demuestra a continuación:

In [4]:
expresiones_validas = []                        # Lista donde guardaremos solo las expresiones cuyo resultado sea un número entero

for expresion in expresiones:                   # Recorremos una a una todas las expresiones generadas antes
    resultado = eval(expresion)

    if resultado.is_integer():                  # is_integer() es un método de los float: devuelve True si el decimal es .0
        expresiones_validas.append(
            (expresion, int(resultado))         # Si es entero, guardamos una tupla: (la expresión en texto, el resultado como int)
        )

print(
    f"Teniendo en cuenta la restricción de que deben de ser enteros hay {len(expresiones_validas)} combinaciones posibles."
)

Teniendo en cuenta la restricción de que deben de ser enteros hay 90000 combinaciones posibles.


Modelo para el espacio de soluciones<br>
(*) ¿Cual es la estructura de datos que mejor se adapta al problema? Argumentalo.(Es posible que hayas elegido una al principio y veas la necesidad de cambiar, arguentalo)


Respuesta

La lista nos permite conservar todas las expresiones válidas, incluso cuando varias de ellas producen el mismo resultado. Esto es importante porque el problema no solo pide conocer qué enteros pueden obtenerse, sino también encontrar las expresiones que los generan.

Sin embargo, para comprobar qué resultados enteros distintos son alcanzables, una lista no es la estructura más eficiente: comprobar si un valor pertenece a una lista requiere recorrerla entera, O(n). En este caso es preferible utilizar un **conjunto** (`set`):

    enteros_alcanzables = set(resultados)

El conjunto elimina automáticamente los valores repetidos y permite comprobar de forma eficiente si un número pertenece al conjunto de resultados alcanzables. Además, facilita comparar los resultados obtenidos con el intervalo entre el mínimo y el máximo mediante diferencia de conjuntos (`rango_completo - enteros_alcanzables`), una operación que con listas sería mucho más costosa.

Por tanto, se utilizan dos estructuras complementarias:

- Una **lista de tuplas** `(expresion, resultado)` para almacenar todas las expresiones válidas junto con el valor que producen.
- Un **conjunto** para almacenar los resultados enteros distintos y comprobar de forma eficiente cuáles son alcanzables.


Según el modelo para el espacio de soluciones<br>
(*)¿Cual es la función objetivo?

(*)¿Es un problema de maximización o minimización?

Respuesta

In [5]:
def cumple_restriccion(expresion):
    """Función de restricción: True si el resultado de la expresión es un entero.

    No es la función objetivo del problema: es el
    filtro que determina si una expresión pertenece al espacio de
    soluciones válidas.
    """
    resultado = eval(expresion)              # Evaluamos la expresión
    return resultado.is_integer()            # Comprobamos si ese número es entero y devolvemos True/False


def valor(expresion):
    """Función objetivo de los subproblemas de mínimo/máximo: valor entero
    obtenido al evaluar la expresión.
    """
    return int(eval(expresion))              # Evaluamos la expresión y convertimos el resultado (float) a int

La función objetivo no es una función que busque maximizar o minimizar en sentido estricto, sino una función de evaluación sujeta a la restricción de que el resultado sea un número entero (`cumple_restriccion`).

Por ello, el problema tal como se plantea (encontrar todas las expresiones válidas) es un problema de búsqueda, no un problema de maximización ni de minimización.

Aun asi, el propio enunciado introduce dos subproblemas de optimización derivados de ese conjunto de soluciones válidas: encontrar el valor mínimo y el valor máximo alcanzables. Para esos dos subproblemas sí hay una función objetivo clara (`valor(expresion)`, el valor entero resultante de la expresión) y un sentido de optimización explícito: minimizarla en un caso y maximizarla en el otro.

Diseña un algoritmo para resolver el problema por fuerza bruta

Respuesta

In [6]:
def resolver_fuerza_bruta(
    cifras=('1','2','3','4','5','6','7','8','9'),
    operadores=('+','-','*','/')
):
    """
    Genera todas las expresiones posibles mediante
    fuerza bruta y devuelve aquellas cuyo resultado
    es un número entero.
    """

    expresiones_validas = []                                               # Aquí acumularemos las tuplas (expresion, valor) de las soluciones válidas

    for secuencia_cifras in itertools.permutations(cifras, 5):
        for secuencia_operadores in itertools.permutations(operadores):

            partes = []                                                    # Lista auxiliar donde iremos metiendo cifras y operadores en orden

            for cifra, operador in zip(                                    # Recorremos las 4 primeras cifras junto con los 4 operadores, a la vez
                secuencia_cifras[:-1],
                secuencia_operadores
            ):
                partes.append(cifra)                                       # Añadimos la cifra
                partes.append(operador)                                    # Añadimos el operador que va justo después

            partes.append(secuencia_cifras[-1])                            # Añadimos la última cifra
            expresion = ''.join(partes)                                    # Concatenamos todos los elementos de la lista en un solo string

            if cumple_restriccion(expresion):                              # Si el resultado de esta expresión es entero, guardamos la expresión junto con su valor numérico
                expresiones_validas.append(
                    (expresion, valor(expresion))
                )

    return expresiones_validas                                             # Devolvemos la lista completa de soluciones válidas encontradas

In [7]:
def analizar_resultados(expresiones_validas):
    resultados = [resultado for _, resultado in expresiones_validas]       # Nos quedamos solo con el segundo elemento de cada tupla (el resultado numérico)

    valor_min = min(resultados)                                            # Cogemos el mínimo y máximo de la lista "resultados"
    valor_max = max(resultados)

    enteros_alcanzables = set(resultados)                                  # Convertimos la lista en un conjunto, eliminando los valores repetidos
    rango_completo = set(range(valor_min, valor_max + 1))                  # Generamos todos los enteros del intervalo
    enteros_faltantes = sorted(rango_completo - enteros_alcanzables)       # Calculamos y ordenamos los enteros del rango que NO se han conseguido alcanzar

    return {
        "min": valor_min,
        "max": valor_max,
        "total_enteros_posibles_en_rango": len(rango_completo),
        "total_enteros_alcanzados": len(enteros_alcanzables),
        "enteros_faltantes": enteros_faltantes,
    }

In [8]:
expresiones_validas_fb = resolver_fuerza_bruta()                          # Ejecutamos el algoritmo de fuerza bruta y guardamos todas las soluciones válidas
analisis = analizar_resultados(expresiones_validas_fb)                    # Analizamos esas soluciones

print(f"El algoritmo ha encontrado {len(expresiones_validas_fb)} expresiones válidas.")
print(f"Valor mínimo alcanzable: {analisis['min']}")
print(f"Valor máximo alcanzable: {analisis['max']}")
print(f"Enteros en el rango [{analisis['min']}, {analisis['max']}]: {analisis['total_enteros_posibles_en_rango']}")
print(f"Enteros realmente alcanzados: {analisis['total_enteros_alcanzados']}")
print(f"Enteros del rango que NO se pueden obtener: {len(analisis['enteros_faltantes'])}")

El algoritmo ha encontrado 90000 expresiones válidas.
Valor mínimo alcanzable: -69
Valor máximo alcanzable: 77
Enteros en el rango [-69, 77]: 147
Enteros realmente alcanzados: 147
Enteros del rango que NO se pueden obtener: 0


Calcula la complejidad del algoritmo por fuerza bruta

Respuesta

- **Generar las secuencias de cifras**: son variaciones sin repetición de n elementos tomados de k en k, V(n,k) = n! / (n-k)!. Para n=9, k=5: 9·8·7·6·5 = 15.120. Para k fijo, V(n,k) = O(n^k).
- **Generar las secuencias de operadores**: como se usan siempre los m operadores, son todas sus permutaciones, m! = 4! = 24.
- **Por cada combinación** (cifras, operadores) se construye la cadena de texto (coste O(k), es decir O(1) al ser k constante) y se evalúa con `eval()` (coste O(1), porque la longitud de la expresión no depende de n).

En este problema concreto, k=5 (cifras a elegir) y m=4 (operadores) son constantes fijadas por el enunciado, por lo que no crecen con n. La única variable que realmente crece es n (el tamaño del conjunto de cifras disponible), así que la complejidad, dejando k y m fijos, es:

T(n) = O( V(n,k) · m! · O(1) ) = O( n! / (n-k)! ) = O(n^k) = **O(n^5)**, polinómica en n.


(*)Diseña un algoritmo que mejore la complejidad del algortimo por fuerza bruta. Argumenta porque crees que mejora el algoritmo por fuerza bruta

Respuesta

La fuerza bruta genera las 362.880 cadenas de texto y las evalúa una a una con `eval()`. Se puede hacer mucho mejor si nos fijamos en la estructura algebraica de las expresiones.

Por la precedencia de operadores, toda expresión se descompone en *términos* (grupos de cifras unidos por `*` y `/`) separados por `+` y `-`. Como `+` y `-` aparecen exactamente una vez cada uno, toda expresión tiene exactamente tres términos y su valor es:

$$valor = t_1 + t_2 - t_3$$

La suma es conmutativa, así que lo único que importa es qué término se resta, no el orden en que se escriben los términos. Esto reduce las $4! = 24$ ordenaciones de operadores de la fuerza bruta (y las reordenaciones equivalentes de las cifras) a solo 3 asignaciones de signo por descomposición.

Las 5 cifras solo pueden repartirse entre los tres términos de dos formas, porque `*` y `/` son los operadores internos de los términos:

* **Caso A (3,1,1)**: un término de 3 cifras que contiene `*` y `/`, y dos cifras sueltas. Las formas posibles del término son `x*y/z` y `x/y*z`; como `x/y*z` vale lo mismo que `x*z/y`, basta enumerar `x*y/z` sobre ternas ordenadas.
* **Caso B (2,2,1)**: un término producto `x*y`, un término división `p/q` y una cifra suelta.

Los términos sin división son siempre enteros, de modo que el resultado es entero si y solo si el término que contiene la división lo es. Esta condición ($z$ divide a $x \cdot y$, o $q$ divide a $p$) se comprueba antes de construir la expresión y poda de golpe todas las ramas que no pueden dar un entero. Cada candidato que sobrevive a la poda se evalúa con 2 o 3 operaciones aritméticas enteras exactas, sin construir strings ni pasar por `eval()`, y al estar implementado como generador admite parada temprana cuando se busca una cantidad dada, que es la formulación original del problema.

In [9]:
def generar_soluciones(numeros=(1, 2, 3, 4, 5, 6, 7, 8, 9)):
    """Generador de descomposiciones  valor = t1 + t2 - t3  con resultado entero.

    Produce tuplas (valor, expresion). La expresión se construye ya en el
    formato del enunciado: cifra-operador-cifra... usando cada operador una vez.
    """

    # Caso A: término de 3 números con * y / juntos (x*y/z), más dos números sueltos
    for x, y, z in itertools.permutations(numeros, 3):
        if (x * y) % z != 0:                                  # Poda: si z no divide a x*y el resultado nunca será entero
            continue
        t = (x * y) // z                                      # Valor exacto del término, en aritmética entera
        resto = [d for d in numeros if d not in (x, y, z)]
        for u, v in itertools.combinations(resto, 2):
            yield t + u - v, f"{x}*{y}/{z}+{u}-{v}"           # Se resta la cifra v
            yield t + v - u, f"{x}*{y}/{z}+{v}-{u}"           # Se resta la cifra u
            yield u + v - t, f"{u}+{v}-{x}*{y}/{z}"           # Se resta el término con la división

    # Caso B: un producto x*y, una división p/q y un número suelto s
    for x, y in itertools.combinations(numeros, 2):           # El producto no depende del orden: combinaciones
        m = x * y
        resto = [d for d in numeros if d not in (x, y)]
        for p, q in itertools.permutations(resto, 2):         # La división sí depende del orden: permutaciones
            if p % q != 0:                                    # Poda: la división debe ser exacta
                continue
            d = p // q
            for s in resto:
                if s in (p, q):
                    continue
                yield m + d - s, f"{x}*{y}+{p}/{q}-{s}"       # Se resta la cifra suelta
                yield m + s - d, f"{x}*{y}+{s}-{p}/{q}"       # Se resta la división
                yield d + s - m, f"{p}/{q}+{s}-{x}*{y}"       # Se resta el producto


def resolver_mejorado(numeros=(1, 2, 3, 4, 5, 6, 7, 8, 9)):
    """Devuelve ({valor: expresión de ejemplo}, nº de candidatos evaluados)."""
    soluciones = {}
    candidatos = 0
    for valor_entero, expresion in generar_soluciones(numeros):
        candidatos += 1
        soluciones.setdefault(valor_entero, expresion)        # Nos quedamos con la primera expresión de cada valor
    return soluciones, candidatos


def cotas_del_rango(numeros):
    """Cotas inferior y superior del valor de cualquier expresión, calculadas en O(n).

    Como solo hay un '*', ningún término puede superar el producto de los dos
    números mayores. Las cotas son algo más amplias que el rango real, así que
    nunca descartan un objetivo alcanzable.
    """
    tercero, segundo, primero = sorted(numeros)[-3:]          # Los tres números mayores
    producto_maximo = primero * segundo
    return 2 - producto_maximo, producto_maximo + tercero - 1


def buscar_objetivo(objetivo, numeros=(1, 2, 3, 4, 5, 6, 7, 8, 9)):
    """Devuelve una expresión cuyo valor es `objetivo`, o None si no existe.

    Los objetivos fuera de las cotas se descartan sin generar ningún candidato.
    Para el resto se recorre el generador, deteniéndose en cuanto lo encuentra.
    """
    cota_inferior, cota_superior = cotas_del_rango(numeros)
    if not (cota_inferior <= objetivo <= cota_superior):
        return None

    for valor_entero, expresion in generar_soluciones(numeros):
        if valor_entero == objetivo:
            return expresion
    return None

In [10]:
import time

t0 = time.perf_counter()
expresiones_validas_fb = resolver_fuerza_bruta()                          # La ejecutamos de nuevo para poder medir su tiempo
t_fb = time.perf_counter() - t0

t0 = time.perf_counter()
soluciones_mejorado, candidatos = resolver_mejorado()
t_mejorado = time.perf_counter() - t0

valores_fb = {resultado for _, resultado in expresiones_validas_fb}

print(f"Fuerza bruta : {len(valores_fb)} valores distintos | 362.880 candidatos | {t_fb:.2f} s")
print(f"Mejorado     : {len(soluciones_mejorado)} valores distintos | {candidatos} candidatos | {t_mejorado:.4f} s")
print(f"¿Encuentran los mismos valores? {valores_fb == set(soluciones_mejorado)}")
print(f"Aceleración: x{t_fb / t_mejorado:.0f}")
print()
minimo, maximo = min(soluciones_mejorado), max(soluciones_mejorado)
print(f"Valor mínimo: {minimo}  con la expresión  {soluciones_mejorado[minimo]} = {eval(soluciones_mejorado[minimo])}")
print(f"Valor máximo: {maximo}  con la expresión  {soluciones_mejorado[maximo]} = {eval(soluciones_mejorado[maximo])}")
print()
objetivo_ejemplo = 4                                                      # El ejemplo del enunciado
print(f"Búsqueda con parada temprana del objetivo {objetivo_ejemplo}: {buscar_objetivo(objetivo_ejemplo)}")

Fuerza bruta : 147 valores distintos | 362.880 candidatos | 5.26 s
Mejorado     : 147 valores distintos | 11250 candidatos | 0.0074 s
¿Encuentran los mismos valores? True
Aceleración: x713

Valor mínimo: -69  con la expresión  4/2+1-8*9 = -69.0
Valor máximo: 77  con la expresión  8*9/1+7-2 = 77.0

Búsqueda con parada temprana del objetivo 4: 1*4/2+5-3


(*)Calcula la complejidad del algoritmo

Respuesta

El algoritmo mejorado recorre dos casos, y en cada uno hay un bucle interno al que solo se llega si el candidato supera la poda:

* **Caso A**: se recorren las $V(n,3)=\Theta(n^3)$ ternas, y las que cumplen que $z$ divide a $x\cdot y$ entran en un bucle de $C(n-3,2)=\Theta(n^2)$ pares.
* **Caso B**: se recorren $C(n,2)\cdot V(n-2,2)=\Theta(n^4)$ combinaciones de producto y división, y las que tienen división exacta entran en un bucle de $\Theta(n)$ para la cifra suelta.

Si la poda dejase pasar una fracción constante de los candidatos, el coste sería $\Theta(n^5)$, el mismo que la fuerza bruta, y la ganancia se reduciría a factores constantes. Pero la divisibilidad no filtra una fracción constante: los pares $(p,q)$ en los que $q$ divide a $p$ entre los $n$ primeros números son $\sum_{q=1}^{n}\lfloor n/q\rfloor=\Theta(n\log n)$ y no $\Theta(n^2)$, y las ternas en las que $z$ divide a $x\cdot y$ son del orden de $n^2$ y no de $n^3$. Sustituyendo, el coste total crece como $n^4$ salvo factores logarítmicos, frente al $n^5$ de la fuerza bruta.

Como el cociente entre ambos crece sin límite, la mejora es de clase asintótica y no de factores constantes. La celda siguiente lo comprueba: si solo se ganaran constantes, la razón entre los candidatos de un algoritmo y otro sería fija, y en cambio pasa de 32 con n=9 a 154 con n=144.

Conviene precisar el peor caso. Esto vale para las cifras del enunciado y, en general, para conjuntos de números de magnitud acotada. Con un conjunto elegido a propósito, como las potencias de 2, todas las divisiones son exactas, la poda no descarta nada y el coste vuelve a ser $\Theta(n^5)$.

Por este camino tampoco se puede bajar mucho más, porque las expresiones con resultado entero son ya del orden de $n^4$ (para n=9, 90.000 de las 362.880 candidatas): el algoritmo cuesta prácticamente lo que ocupa su salida, mientras que la fuerza bruta paga $\Theta(n^5)$ en cualquier caso. Sí quedaría margen si solo se pidieran los valores enteros distintos y no las expresiones que los producen, porque esos valores están acotados por el rango $[-n^2, n^2]$ y son únicamente $\Theta(n^2)$.

En la versión generalizada con $m$ operadores se gana además un factor factorial: la fuerza bruta cuesta $O(m!\cdot n^{m+1})$ por las ordenaciones de los operadores, mientras que aprovechar la conmutatividad de la suma reduce las $4!=24$ ordenaciones a solo 3 asignaciones de signo por descomposición.

En la búsqueda de una cantidad concreta, `buscar_objetivo` descarta en $O(n)$ los objetivos fuera de las cotas sin generar ningún candidato, y para el resto se detiene en la primera coincidencia.

La descomposición en términos también permite razonar el máximo y el mínimo a mano: el máximo maximiza $t_1 + t_2$ y minimiza $t_3$ ($8 \cdot 9 + 7 - 6/3 = 77$) y el mínimo al revés ($6/3 + 1 - 8 \cdot 9 = -69$), que coincide con lo obtenido por búsqueda exhaustiva.

In [11]:
import math

def contar_candidatos(numeros):
    """Cuenta los candidatos que recorre `generar_soluciones`, sin construir las expresiones.

    Repite sus mismos bucles y sus mismas podas, para poder medir el crecimiento
    con valores de n en los que generar los strings sería inviable.
    """
    n = len(numeros)
    total = 0

    for x, y, z in itertools.permutations(numeros, 3):                    # Caso A
        if (x * y) % z == 0:
            total += 3 * math.comb(n - 3, 2)

    for x, y in itertools.combinations(numeros, 2):                       # Caso B
        resto = [d for d in numeros if d not in (x, y)]
        for p, q in itertools.permutations(resto, 2):
            if p % q == 0:
                total += 3 * (len(resto) - 2)

    return total


assert contar_candidatos(tuple(range(1, 10))) == resolver_mejorado()[1]   # Cuenta lo mismo que el generador

filas = [f"{'n':>5} {'mejorado':>16} {'fuerza bruta':>20} {'razón':>8} {'pendiente':>10}"]
anterior = None

for n in (9, 18, 36, 72, 144):
    mejorado = contar_candidatos(tuple(range(1, n + 1)))
    fuerza_bruta = math.perm(n, 5) * 24                                   # V(n,5)·4!, sin poda posible

    if anterior is None:
        pendiente = "-"
    else:
        n_anterior, mejorado_anterior = anterior                          # Pendiente log-log: exponente empírico de n
        pendiente = f"{math.log(mejorado / mejorado_anterior) / math.log(n / n_anterior):.2f}"

    filas.append(f"{n:>5} {mejorado:>16,} {fuerza_bruta:>20,} {fuerza_bruta / mejorado:>8.0f} {pendiente:>10}")
    anterior = (n, mejorado)

print("\n".join(filas))

    n         mejorado         fuerza bruta    razón  pendiente
    9           11,250              362,880       32          -
   18          568,890           24,675,840       43       5.66
   36       17,303,616        1,085,736,960       63       4.93
   72      414,538,200       40,295,646,720       97       4.58
  144    8,982,371,160    1,385,304,560,640      154       4.44


Según el problema (y tenga sentido), diseña un juego de datos de entrada aleatorios

Respuesta

Los datos de entrada de este problema son el conjunto de cifras disponibles (del 1 al 9) y los 4 operadores, que el enunciado fija. Para poder generar juegos de datos aleatorios generalizamos el conjunto de números: en lugar de las cifras del 1 al 9 usamos un conjunto aleatorio de $n$ números enteros positivos distintos (por ejemplo entre 1 y 20). Las funciones `generar_soluciones`, `resolver_mejorado` y `buscar_objetivo` ya están parametrizadas para admitir cualquier conjunto de números, así que se aplican sin cambios.

Generamos además objetivos aleatorios para probar la búsqueda de una cantidad dada.

In [12]:
import random

def generar_datos_aleatorios(n_numeros=9, valor_maximo=20, semilla=None):
    """Genera un conjunto aleatorio de n números enteros distintos entre 1 y valor_maximo.

    Hace el papel del conjunto de cifras 1-9 del enunciado. Se exige n >= 5
    porque cada expresión necesita 5 números distintos.
    """
    if n_numeros < 5:
        raise ValueError("Se necesitan al menos 5 números para construir una expresión")
    rng = random.Random(semilla)                              # Con semilla fija los resultados son reproducibles
    return tuple(sorted(rng.sample(range(1, valor_maximo + 1), n_numeros)))


juegos_de_datos = [generar_datos_aleatorios(n_numeros=9, valor_maximo=20, semilla=s) for s in (1, 2, 3)]

for datos in juegos_de_datos:
    print(datos)

(3, 4, 5, 8, 9, 13, 14, 15, 19)
(2, 3, 5, 6, 11, 12, 13, 17, 19)
(5, 8, 10, 11, 12, 15, 18, 19, 20)


Aplica el algoritmo al juego de datos generado

Respuesta

In [13]:
for semilla, datos in zip((1, 2, 3), juegos_de_datos):
    soluciones, candidatos = resolver_mejorado(datos)
    minimo, maximo = min(soluciones), max(soluciones)
    rango = maximo - minimo + 1

    print(f"Juego de datos {datos}")
    print(f"  Candidatos evaluados: {candidatos}")
    print(f"  Valor mínimo: {minimo}  ->  {soluciones[minimo]}")
    print(f"  Valor máximo: {maximo}  ->  {soluciones[maximo]}")
    print(f"  Enteros alcanzados: {len(soluciones)} de los {rango} del rango [{minimo}, {maximo}]")

    objetivo = random.Random(semilla).randint(minimo, maximo) # Objetivo aleatorio reproducible dentro del rango
    expresion = buscar_objetivo(objetivo, datos)
    if expresion is not None:
        print(f"  Objetivo aleatorio {objetivo}: {expresion} = {eval(expresion)}")
    else:
        print(f"  Objetivo aleatorio {objetivo}: no alcanzable con estos números")
    print()

Juego de datos (3, 4, 5, 8, 9, 13, 14, 15, 19)
  Candidatos evaluados: 4050
  Valor mínimo: -280  ->  8/4+3-15*19
  Valor máximo: 297  ->  15*19+14-8/4
  Enteros alcanzados: 459 de los 578 del rango [-280, 297]
  Objetivo aleatorio -143: 15/3+4-8*19 = -143.0

Juego de datos (2, 3, 5, 6, 11, 12, 13, 17, 19)
  Candidatos evaluados: 4725
  Valor mínimo: -319  ->  6/3+2-17*19
  Valor máximo: 334  ->  17*19+13-6/3
  Enteros alcanzados: 473 de los 654 del rango [-319, 334]
  Objetivo aleatorio -262: no alcanzable con estos números

Juego de datos (5, 8, 10, 11, 12, 15, 18, 19, 20)
  Candidatos evaluados: 6210
  Valor mínimo: -370  ->  10/5+8-19*20
  Valor máximo: 396  ->  19*20+18-10/5
  Enteros alcanzados: 584 de los 767 del rango [-370, 396]
  Objetivo aleatorio -127: 10/5+15-8*18 = -127.0



Enumera las referencias que has utilizado(si ha sido necesario) para llevar a cabo el trabajo

Respuesta

- Brassard, G. y Bratley, P. (1997). *Fundamentos de algoritmia*. Prentice Hall. ISBN 9788489660007.
- Guerequeta, R. y Vallecillo, A. (2000). *Técnicas de diseño de algoritmos*. Universidad de Málaga. http://www.lcc.uma.es/~av/Libro/indice.html
- Duarte, A. (2008). *Metaheurísticas*. Madrid: Dykinson.
- Documentación oficial de Python: módulo `itertools` (https://docs.python.org/3/library/itertools.html), función `eval` y método `float.is_integer` (https://docs.python.org/3/library/stdtypes.html).
- Material de la asignatura 03MIAR Algoritmos de Optimización (VIU): videoconferencia VC4, Problemas del Trabajo Práctico.

Describe brevemente las lineas de como crees que es posible avanzar en el estudio del problema. Ten en cuenta incluso posibles variaciones del problema y/o variaciones al alza del tamaño

Respuesta

El siguiente paso natural sería relajar las restricciones del enunciado: permitir repetir operadores, usar las 9 cifras (expresiones más largas, con número variable de términos) o admitir paréntesis, en cuyo caso el espacio pasa a ser el de los árboles de expresión y convendría una búsqueda recursiva con memoización de subexpresiones, al estilo del problema *Countdown*. Para objetivos no enteros bastaría sustituir la poda de divisibilidad por fracciones exactas (`fractions.Fraction`).

En cuanto al tamaño, con $n$ números y $m$ operadores el espacio crece como $O(n^{m+1} \cdot m!)$ y la enumeración completa deja de ser viable enseguida. Ahí entrarían cotas para ramificación y poda, programación dinámica sobre subconjuntos con máscaras de bits y, cuando ni eso alcance, las metaheurísticas vistas en la asignatura (búsqueda local, recocido simulado, algoritmos genéticos) usando la distancia al objetivo como fitness.